In [ ]:
%pip install -q jiwer pandas

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not on Colab, skipping Drive mount")

In [ ]:
from pathlib import Path

import pandas as pd
from jiwer import process_words

EXPECTED_SAMPLES = 3295

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/DhwaniLab/results"),
    Path("results"),
    Path("../results"),
    Path("/kaggle/input"),
]

MODELS = {
    "indicwhisper": "indicvoices_telugu_valid.csv",
    "indicconformer": "indicvoices_telugu_valid.csv",
}

def resolve(model_name, filename):
    for root in SEARCH_ROOTS:
        candidate = root / model_name / filename
        if candidate.exists():
            return candidate
    return None

paths = {}

for model_name, filename in MODELS.items():
    path = resolve(model_name, filename)
    paths[model_name] = path
    print(f"{model_name:16}", path if path else "NOT FOUND")

missing = [m for m, p in paths.items() if p is None]

if missing:
    raise FileNotFoundError(
        f"Could not locate CSVs for: {missing}. "
        f"Mount Drive or place them under one of {[str(r) for r in SEARCH_ROOTS]}"
    )

In [ ]:
def normalize_for_wer(text):
    return " ".join(str(text).strip().split())

frames = {}

for model_name, path in paths.items():
    df = pd.read_csv(path)

    df["index"] = df["index"].astype(int)
    df["reference"] = df["reference"].fillna("").map(normalize_for_wer)
    df["prediction"] = df["prediction"].fillna("").map(normalize_for_wer)

    frames[model_name] = df

    print(f"{model_name:16} rows={len(df):5}  columns={list(df.columns)}")

In [ ]:
for model_name, df in frames.items():
    n = len(df)
    duplicates = df["index"].duplicated().sum()
    unique_idx = df["index"].nunique()
    lo, hi = df["index"].min(), df["index"].max()
    expected = set(range(EXPECTED_SAMPLES))
    gaps = sorted(expected - set(df["index"].tolist()))
    empty_pred = (df["prediction"].str.len() == 0).sum()
    empty_ref = (df["reference"].str.len() == 0).sum()

    print(f"--- {model_name} ---")
    print(f"  rows              : {n}")
    print(f"  expected          : {EXPECTED_SAMPLES}")
    print(f"  complete          : {n == EXPECTED_SAMPLES}")
    print(f"  unique indices    : {unique_idx}")
    print(f"  duplicate indices : {duplicates}")
    print(f"  index range       : {lo} to {hi}")
    print(f"  missing indices   : {len(gaps)}")

    if gaps:
        print(f"  first missing     : {gaps[:10]}")

    print(f"  empty references  : {empty_ref}")
    print(f"  empty predictions : {empty_pred}")
    print()

In [ ]:
names = list(frames)

a, b = names[0], names[1]

refs = frames[a][["index", "reference"]].merge(
    frames[b][["index", "reference"]],
    on="index",
    how="inner",
    suffixes=(f"_{a}", f"_{b}")
)

refs["match"] = refs[f"reference_{a}"] == refs[f"reference_{b}"]

mismatches = refs[~refs["match"]]

print("Overlapping indices :", len(refs))
print("References matching :", int(refs["match"].sum()))
print("References differing:", len(mismatches))

if len(mismatches):
    print()
    for row in mismatches.head(5).to_dict("records"):
        print("index:", row["index"])
        print(f"  {a}:", row[f"reference_{a}"])
        print(f"  {b}:", row[f"reference_{b}"])
        print()

In [ ]:
common_indices = sorted(
    set(frames[a]["index"]).intersection(set(frames[b]["index"]))
)

print("Common indices:", len(common_indices))

results = []

for model_name, df in frames.items():
    subset = df[df["index"].isin(common_indices)].sort_values("index")

    outcome = process_words(
        subset["reference"].tolist(),
        subset["prediction"].tolist()
    )

    reference_words = (
        outcome.hits
        + outcome.substitutions
        + outcome.deletions
    )

    results.append({
        "model": model_name,
        "samples": len(subset),
        "reference_words": reference_words,
        "hits": outcome.hits,
        "substitutions": outcome.substitutions,
        "deletions": outcome.deletions,
        "insertions": outcome.insertions,
        "corpus_wer": outcome.wer,
        "corpus_wer_percent": outcome.wer * 100,
    })

comparison_df = pd.DataFrame(results).sort_values("corpus_wer")

comparison_df

In [ ]:
for row in comparison_df.to_dict("records"):
    print(f"{row['model']:16} WER {row['corpus_wer_percent']:6.2f}%  "
          f"S={row['substitutions']:6} D={row['deletions']:6} I={row['insertions']:6} "
          f"of {row['reference_words']} words")

best = comparison_df.iloc[0]
worst = comparison_df.iloc[-1]

gap = worst["corpus_wer_percent"] - best["corpus_wer_percent"]

print()
print(f"Lower WER: {best['model']} by {gap:.2f} percentage points")